### Extract spike timeseries aligned to specific event for every unit

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *
import scipy.io
from config import dir_config, ephys_config
from src.utils import ephys_utils
import pickle
from scipy.stats import ttest_rel

compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Function to align and convolve spike trains
def get_spike_counts(cluster_spike_time, timestamps, trial_info, alignment_settings, sampling_rate=30):
    result = {}

    # Prepare arrays for all alignment events
    for epoch in alignment_settings.keys():
        n_trials = len(trial_info)
        spike_counts = np.full((n_trials, 1), np.nan, dtype=np.int16)
        duration = (alignment_settings[epoch]["end_time_ms"] - alignment_settings[epoch]["start_time_ms"]) / 1000  # in seconds
        # Iterate through trials
        for idx_trial, trial_num in enumerate(trial_info.index):
            if np.isnan(trial_info.reaction_time[trial_num]):
                continue

            aligned_event_time = timestamps.loc[trial_num, alignment_settings[epoch]["event"]]
            start_timestamp = aligned_event_time + (alignment_settings[epoch]["start_time_ms"]) * sampling_rate
            end_timestamp = aligned_event_time + (alignment_settings[epoch]["end_time_ms"]) * sampling_rate

            # Filter spike times from start_timestamp to end_timestamp
            temp_spike_times = cluster_spike_time[(cluster_spike_time >= start_timestamp) & (cluster_spike_time <= end_timestamp)]
            spike_counts[idx_trial] = len(temp_spike_times)

        # Store results
        result[epoch] = spike_counts / duration  # Convert to firing rate (spikes per second)

    return result


In [ ]:

# Load neuron metadata
neuron_metadata = pd.read_csv(Path(compiled_dir, "neuron_metadata.csv"), index_col=None)
spike_rate_neuron_wise = {neuron: {} for neuron in neuron_metadata.neuron_id}

# Main loop for each neuron
for neuron in neuron_metadata.neuron_id:
    session_name = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron].values[0]
    cluster_id = neuron_metadata.cluster[neuron_metadata.neuron_id == neuron].values[0]

    # Load required data
    timestamps_path = Path(compiled_dir, session_name, f"{session_name}_timestamps.csv")
    trial_info_path = Path(compiled_dir, session_name, f"{session_name}_trial.csv")
    spike_times_path = Path(compiled_dir, session_name, "spike_times.npy")
    spike_clusters_path = Path(compiled_dir, session_name, "spike_clusters.npy")
    spike_times_mat_path = Path(compiled_dir, session_name, "spike_times.mat")
    spike_clusters_mat_path = Path(compiled_dir, session_name, "spike_clusters.mat")

    if not (timestamps_path.is_file() and trial_info_path.is_file()):
        print(f"Missing files for session: {session_name}")
        continue

    timestamps = pd.read_csv(timestamps_path, index_col=None)
    trial_info = pd.read_csv(trial_info_path, index_col=None)

    # Load spike data
    if spike_times_path.is_file() and spike_clusters_path.is_file():
        spike_times = np.load(spike_times_path)
        spike_clusters = np.load(spike_clusters_path)
    elif spike_times_mat_path.is_file() and spike_clusters_mat_path.is_file():
        spike_times = scipy.io.loadmat(spike_times_mat_path)["spike_times"].ravel()
        spike_clusters = scipy.io.loadmat(spike_clusters_mat_path)["spike_clusters"].ravel()
    else:
        print(f"Spike times and clusters not found in {session_name} for neuron {neuron}")
        continue

    # Filter spike times for the current cluster
    cluster_spike_time = spike_times[spike_clusters == cluster_id]
    VGS_trial_info = trial_info[(trial_info.task_type == 2) & ((trial_info.outcome) == 1)]  # VGS trials with correct outcome 
    # only include trials with non-nan reaction time
    VGS_trial_info = VGS_trial_info[~VGS_trial_info.reaction_time.isna()]
    # only include trials with toRF choice
    VGS_trial_info = VGS_trial_info[VGS_trial_info.choice == 1]
    
    # Get aligned and convolved spike trains
    results = get_spike_counts(cluster_spike_time, timestamps, VGS_trial_info, ephys_config.alignment_settings_VGS)

    spike_rate_neuron_wise[neuron] = results
    

In [ ]:

# Load neuron metadata
neuron_metadata = pd.read_csv(Path(compiled_dir, "neuron_metadata.csv"), index_col=None)
delay_duration_JP, delay_duration_TZ = [], []
# Main loop for each neuron
for neuron in neuron_metadata.neuron_id:
    session_name = neuron_metadata.session_id[neuron - 1]
    cluster_id = neuron_metadata.cluster[neuron - 1]
    # Load required data
    timestamps_path = Path(compiled_dir, session_name, f"{session_name}_timestamps.csv")
    trial_info_path = Path(compiled_dir, session_name, f"{session_name}_trial.csv")

    if not (timestamps_path.is_file() and trial_info_path.is_file()):
        print(f"Missing files for session: {session_name}")
        continue

    timestamps = pd.read_csv(timestamps_path, index_col=None)
    trial_info = pd.read_csv(trial_info_path, index_col=None)


    VGS_trial_info = trial_info[(trial_info.task_type == 2) & ((trial_info.outcome) == 1)]  # VGS trials with correct outcome

    VGS_timestamps = timestamps[(trial_info.task_type == 2) & ((trial_info.outcome) == 1)]
    delay_duration = VGS_timestamps["go_onset"] - VGS_timestamps["target_onset"]
    if 'JP' in session_name:
        delay_duration_JP.append(delay_duration/30)
    elif 'TZ' in session_name:
        delay_duration_TZ.append(delay_duration/30)


In [ ]:
print(f"Minimum delay duration: {np.min(np.concatenate(delay_duration_JP + delay_duration_TZ))}")

## classification (adpated from Li and Basso 2008)

baseline interval: -200 to 0 ms to target onset <br>
visual interval: 50 – 150 ms beginning at the appearance of the luminance gratings<br>
delay interval: 200 – 400 ms after the onset of the target<br>
saccade interval: -50 to 0 ms from the saccade onset<br>
look at mean discharge rate

Using only correct trials<br>
buildup neurons: <br>
&emsp;1. at least 30 spikes (sp)/s discharge during the delay interval (Munoz and Wurtz,1995)[ignored]<br>
&emsp;2. significantly greater activity in the delay interval compared with the baseline (t test, p < 0.05)<br>
&emsp;3. significantly greater activity(t test, p < 0.05) in the saccade interval compared with the delay interval<br>
&emsp;&emsp;similar to the criteria used by us and others previously (Edelman and Keller, 1998; Pare´ and Wurtz, 2001; McPeek and Keller, 2002; Li and Basso, 2005; Li et al., 2006). 

Visual motor neurons:<br>
&emsp;1. no significant delay activity compared with the baseline. <br>

Visual tonic:<br>
&emsp;1.  had a visual response <br>
&emsp;2.  significantly greater level of activity in the delay interval than the baseline interval (t test, p < 0.05)<br>
&emsp;3.  no significant difference in activity between the saccade and delay intervals<br>
    
Visual phasic:<br>
&emsp;1. a visual response<br>
&emsp;2. no significant delay and saccade interval activities<br>


visual epoch 50-150ms
delay 200-400 ms
same length for epoch
30 sp/s is very abitrary

In [ ]:
def classify_neuron(baseline, visual, delay, saccade):
    """
    Classify neuron type based on spike discharge rates.

    Inputs:
        baseline : array-like, shape (n_trials,)
        visual   : array-like, shape (n_trials,)
        delay    : array-like, shape (n_trials,)
        saccade  : array-like, shape (n_trials,)

    Output:
        neuron_type : str
            'buildup', 'motor', 'visual_motor', 'visual_tonic', 
            'visual_phasic', or 'unknown'
    """
    
    # Paired t-tests
    t_vis, p_vis = ttest_rel(visual, baseline)
    t_delay, p_delay = ttest_rel(delay, baseline)
    t_sacc_delay, p_sacc_delay = ttest_rel(saccade, delay)
    t_sacc_base, p_sacc_base = ttest_rel(saccade, baseline)
    visual_response = (np.mean(visual) > np.mean(baseline) and p_vis < 0.05)
    motor_response = (np.mean(saccade) > np.mean(baseline) and p_sacc_base < 0.05)
    delay_response = (np.mean(delay) > np.mean(baseline) and p_delay < 0.05)
    
    # ----------------- Buildup neurons -----------------
    if visual_response and delay_response and np.mean(saccade) > np.mean(delay) and p_sacc_delay < 0.05:
        return 'buildup'

    # ----------------- Motor neurons -----------------
    if not visual_response and motor_response:
        return 'motor'

    # ----------------- Visual motor neurons -----------------
    if not delay_response and visual_response and motor_response:
        return 'visual_motor'
    
    # ----------------- Visual tonic neurons -----------------
    if visual_response and delay_response and p_sacc_delay >= 0.05:
        return 'visual_tonic'

    # ----------------- Visual phasic neurons -----------------
    if visual_response and not delay_response and not p_sacc_base < 0.05:
        return 'visual_phasic'
    
    print(f"p-values: visual: {p_vis}, delay: {p_delay}, saccade (delay): {p_sacc_delay}, saccade (base): {p_sacc_base}")
    print(f"mean rates: baseline {np.mean(baseline)}, visual {np.mean(visual)}, delay {np.mean(delay)}, saccade {np.mean(saccade)}")
    print(f"responses: visual {visual_response}, delay {delay_response}, motor {motor_response}")
    return 'unknown'


In [ ]:

for neuron in neuron_metadata.neuron_id:
    delay_data = spike_rate_neuron_wise[neuron]['delay']
    baseline_data = spike_rate_neuron_wise[neuron]['baseline']
    visual_data = spike_rate_neuron_wise[neuron]['visual']
    response_data = spike_rate_neuron_wise[neuron]['response']
    
    neuron_metadata.loc[neuron_metadata.neuron_id == neuron, 'classification'] = classify_neuron(baseline_data, visual_data, delay_data, response_data)
    if neuron_metadata.classification[neuron_metadata.neuron_id == neuron].values[0] == 'unknown':
        print(f"Neuron {neuron} classified as unknown.\n")

    


In [ ]:

visual_tonic_suppressing = [131,167]
unsure_neurons = [57,133]
# 112,114,115 lost in VGS too
# 18, 140 lost in VGS
# 92 prob lost

In [ ]:
neuron_metadata.loc[neuron_metadata.classification == 'unknown']

In [ ]:
# Load neuron metadata
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"), index_col=None)

In [ ]:
total_count = len(neuron_metadata)
cell_types = neuron_metadata.classification.unique()

In [ ]:
neuron_metadata

In [ ]:
counts = neuron_metadata.classification.value_counts()
labels = counts.index.str.replace('_', ' ', regex=False)

In [ ]:
plt.figure(figsize=(2,4))
plt.bar([0], total_count, color='none')
bottom = 0
for ct, c in zip(labels, counts):
    plt.bar([0], c, bottom=bottom, label=ct)
    bottom += c
# plt.legend(loc='upper right')
plt.xticks([])
plt.tight_layout()

In [ ]:
plt.figure(figsize=(6,6))
counts = neuron_metadata.classification.value_counts()
labels = counts.index.str.replace('_', ' ', regex=False)
def absolute_only(pct, all_vals):
    absolute = int(round(pct/100. * sum(all_vals)))
    return f"{absolute}"

counts.plot.pie(
    labels=labels,                          # show classification types
    autopct=lambda pct: absolute_only(pct, counts),
    startangle=90,
    counterclock=False,
    fontsize=14
)

plt.ylabel('')
plt.show()

In [ ]:
neuron_metadata.to_csv(Path(processed_dir, "neuron_metadata.csv"), index=False)

<br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br><br></br>

In [ ]:
session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_to_exclude = ["210210_GP_JP","241209_GP_TZ"]
session_metadata = session_metadata[~session_metadata["session_id"].isin(session_to_exclude)]

neuron_metadata = pd.read_csv(Path(compiled_dir, "neuron_metadata.csv"))

with open(Path(processed_dir, "glm_hmm_masked_final.pkl"), "rb") as f:
	glm_hmm = pickle.load(f)

with open(Path(processed_dir, f'ephys_neuron_wise.pkl'), 'rb') as f:
    ephys = pickle.load(f)

In [ ]:
data = glm_hmm["data"]
n_trial_back = 1
for session_id in data:
    trial_data = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    GP_trial_data = trial_data[trial_data.task_type == 1].reset_index(drop=True)
    # Get valid indices based on outcomes
    valid_idx = np.where(GP_trial_data.outcome >= 0)[0]
	# First valid trial considering n_trial_back
    first_trial = valid_idx[n_trial_back - 1] + 1
    reaction_time = np.array(GP_trial_data.reaction_time)[first_trial:]
    data[session_id]["reaction_time"] = reaction_time


In [ ]:
import copy
high_confidence_threshold = 0.8
state_occupancy = {}
data_flipped = copy.deepcopy(data)
for idx_session, session_id in enumerate(session_metadata["session_id"]):
    model = glm_hmm["model"]["models"][session_id]
    choices = data[session_id]["choices"].values.reshape(-1, 1)
    input = np.array(data[session_id][["normalized_stimulus", "bias", "prev_choice_1", "prev_target_1"]])
    if data[session_id]["mask"] is None:
        mask = None
    else:
        mask = data[session_id]["mask"]
    mask = np.ones_like(choices, dtype=bool) if mask is None else mask

    posterior_probs = model.expected_states(data=choices, input=input, mask=np.array(mask).reshape(-1, 1))[0]
    biased_idx = (posterior_probs[:, 0] > high_confidence_threshold) & np.array(mask)
    unbiased_idx = (posterior_probs[:, 1] > high_confidence_threshold) & np.array(mask)
    state_occupancy[session_id] = {"biased_state_trials": data[session_id]["trial_num"][biased_idx], "unbiased_state_trials": data[session_id]["trial_num"][unbiased_idx]}

    #flip back to toRF/awayRF for inputs
    if session_metadata["prior_direction"][session_metadata["session_id"] == session_id].values[0] == "awayRF":
        data_flipped[session_id]["choices"] = 1 - data[session_id]["choices"]
        data_flipped[session_id]["stimulus"] = -data[session_id]["stimulus"]
        data_flipped[session_id]["normalized_stimulus"] = -data[session_id]["normalized_stimulus"]
        data_flipped[session_id]["prev_choice_1"] = -data[session_id]["prev_choice_1"]
        data_flipped[session_id]["prev_target_1"] = -data[session_id]["prev_target_1"]

In [ ]:
biased_state_trial_info = {}
unbiased_state_trial_info = {}
for session_id in session_metadata.session_id:
	biased_state_trials = state_occupancy[session_id]["biased_state_trials"]
	biased_state_trial_info[session_id] = data_flipped[session_id].loc[np.isin(data_flipped[session_id]["trial_num"], biased_state_trials)]
	unbiased_state_trials = state_occupancy[session_id]["unbiased_state_trials"]
	unbiased_state_trial_info[session_id] = data_flipped[session_id].loc[np.isin(data_flipped[session_id]["trial_num"], unbiased_state_trials)]

In [ ]:
def plot_PSTH(neuron_id, condition_dict, alignment_dict=ephys_config["alignment_settings_GP"]):
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron_id].values[0]


    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    for align_idx, (alignment) in enumerate(['cue', 'response']):
        for state_idx, trial_info in enumerate([biased_state_trial_info, unbiased_state_trial_info]):
            conditions = {
                # "coh_0_choice_toRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0, 1),
                "coh_6_choice_toRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.06, 1, 1),
                # "coh_20_choice_toRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.2, 1, 1),
                "coh_50_choice_toRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.5, 1, 1),
                # "coh_0_choice_awayRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0, 0),
                "coh_6_choice_awayRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.06, 0, 1),
                # "coh_20_choice_awayRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.2, 0, 1),
                "coh_50_choice_awayRF_corr": ephys_utils.get_trial_num(trial_info[session_id], 0.5, 0, 1),
            }
            for condition in conditions:
                trials = conditions[condition]
                plot_params = condition_dict[condition]
                color =plot_params["biased_color"] if state_idx == 0 else plot_params["unbiased_color"]
                label = condition + ("_biased" if state_idx == 0 else "_unbiased")
                condition_data = ephys_utils.get_neural_data_from_trial_num(ephys[alignment][neuron_id], trials, type="convolved_spike_trains")
                if alignment == "cue":
                    # index = np.where(np.isin(trial_info[session_id]["trial_num"], trials))[0]
                    # crop at the timepoint where less than 50% nan
                    nan_50_timepoints = np.min([np.where(np.mean(np.isnan(condition_data), axis=0) < 0.5)[0][-1], condition_data.shape[1]])
                    ax[align_idx].plot(np.nanmean(condition_data, axis=0)[: nan_50_timepoints], label=label, color=color,linewidth=plot_params['lw'], linestyle=plot_params['ls'])
                elif alignment == "response":
                    # crop at the timepoint where less than 50% nan
                    nan_50_timepoints = np.where(np.mean(np.isnan(condition_data), axis=0) < 0.5)[0][0]
                    ax[align_idx].plot(np.nanmean(condition_data, axis=0)[nan_50_timepoints:], label=label, color=color, linewidth=plot_params['lw'], linestyle=plot_params['ls'])
                else:
                    ax[align_idx].plot(np.nanmean(condition_data, axis=0), label=label, color=color, linewidth=plot_params["lw"], linestyle=plot_params['ls'])
        ylim = ax[align_idx].get_ylim()
        ax[align_idx].vlines(-alignment_dict[alignment]["start_time_ms"], ylim[0], ylim[1], color="black", linestyle="--", linewidth=3)
        ax[align_idx].set_title(alignment)
        if alignment == "response":
            ax[align_idx].set_xticks(
                np.arange(0, alignment_dict[alignment]["end_time_ms"] - alignment_dict[alignment]["start_time_ms"] + 1, 50), labels=np.arange(alignment_dict[alignment]["start_time_ms"], alignment_dict[alignment]["end_time_ms"] + 1, 50)
            )
        else:
            ax[align_idx].set_xticks(
                np.arange(0, alignment_dict[alignment]["end_time_ms"] - alignment_dict[alignment]["start_time_ms"] + 1, 100), labels=np.arange(alignment_dict[alignment]["start_time_ms"], alignment_dict[alignment]["end_time_ms"] + 1, 100)
            )

    ax[-1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    fig.suptitle(f"Neuron ID: {neuron_id} | Session ID: {session_id} | Prior: {session_metadata.prior_direction[session_metadata.session_id==session_id].values[0]}", fontsize=16)

In [ ]:
toRF_neurons = [112, 73, 127, 31, 82, 80]
awayRF_neurons = [109, 140, 137, 2, 153]

In [ ]:
included_neurons = [163,116,113,112,85,73,56,23,20,16,14,12,8,6]
#other ochrence for 85

In [ ]:
# condition_dict = {
# 	"coh_0_choice_toRF_corr": {"index": 0, "color": "red", "linestyle": "-", "opacity": 1},
# 	"coh_6_choice_toRF_corr": {"index": 1, "color": "orange", "linestyle": "-", "opacity": 1},
# 	"coh_20_choice_toRF_corr": {"index": 2, "color": "green", "linestyle": "-", "opacity": 1},
# 	"coh_50_choice_toRF_corr": {"index": 3, "color": "blue", "linestyle": "-", "opacity": 1},
# 	"coh_0_choice_awayRF_corr": {"index": 4, "color": "red", "linestyle": "--", "opacity": 0.3},
# 	"coh_6_choice_awayRF_corr": {"index": 5, "color": "orange", "linestyle": "--", "opacity": 0.3},
# 	"coh_20_choice_awayRF_corr": {"index": 6, "color": "green", "linestyle": "--", "opacity": 0.3},
# 	"coh_50_choice_awayRF_corr": {"index": 7, "color": "blue", "linestyle": "--", "opacity": 0.3},
# }

biased_colors = ["#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
unbiased_colors = ["#F2A448", "#EF8D41", "#EC6A50", "#AC3626"]

condition_dict = {
    "coh_0_choice_toRF_corr": {"index": 0, "biased_color": biased_colors[0], "unbiased_color": unbiased_colors[0], "lw": 3, "ls": '-'},
    "coh_6_choice_toRF_corr": {"index": 1, "biased_color": biased_colors[1], "unbiased_color": unbiased_colors[1], "lw": 3, "ls": '-'},
    "coh_20_choice_toRF_corr": {"index": 2, "biased_color": biased_colors[2], "unbiased_color": unbiased_colors[2], "lw": 3, "ls": '-'},
    "coh_50_choice_toRF_corr": {"index": 3, "biased_color": biased_colors[3], "unbiased_color": unbiased_colors[3], "lw": 3, "ls": '-'},
    "coh_0_choice_awayRF_corr": {"index": 4, "biased_color": biased_colors[0], "unbiased_color": unbiased_colors[0], "lw": 3, "ls": '--'},
    "coh_6_choice_awayRF_corr": {"index": 5, "biased_color": biased_colors[1], "unbiased_color": unbiased_colors[1], "lw": 3, "ls": '--'},
    "coh_20_choice_awayRF_corr": {"index": 6, "biased_color": biased_colors[2], "unbiased_color": unbiased_colors[2], "lw": 3, "ls": '--'},
    "coh_50_choice_awayRF_corr": {"index": 7, "biased_color": biased_colors[3], "unbiased_color": unbiased_colors[3], "lw": 3, "ls": '--'},
}
for neuron in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron].values[0]

    if session_id not in session_metadata["session_id"].values:
        continue
    plot_PSTH(neuron, condition_dict, alignment_dict=ephys_config["alignment_settings_GP"])


In [ ]:
# condition_dict = {
# 	"coh_0_choice_toRF_corr": {"index": 0, "color": "red", "linestyle": "-", "opacity": 1},
# 	"coh_6_choice_toRF_corr": {"index": 1, "color": "orange", "linestyle": "-", "opacity": 1},
# 	"coh_20_choice_toRF_corr": {"index": 2, "color": "green", "linestyle": "-", "opacity": 1},
# 	"coh_50_choice_toRF_corr": {"index": 3, "color": "blue", "linestyle": "-", "opacity": 1},
# 	"coh_0_choice_awayRF_corr": {"index": 4, "color": "red", "linestyle": "--", "opacity": 0.3},
# 	"coh_6_choice_awayRF_corr": {"index": 5, "color": "orange", "linestyle": "--", "opacity": 0.3},
# 	"coh_20_choice_awayRF_corr": {"index": 6, "color": "green", "linestyle": "--", "opacity": 0.3},
# 	"coh_50_choice_awayRF_corr": {"index": 7, "color": "blue", "linestyle": "--", "opacity": 0.3},
# }

biased_colors = ["#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
unbiased_colors = ["#F2A448", "#EF8D41", "#EC6A50", "#AC3626"]

condition_dict = {
    "coh_0_choice_toRF_corr": {"index": 0, "biased_color": biased_colors[0], "unbiased_color": unbiased_colors[0], "lw": 3, "ls": '-'},
    "coh_6_choice_toRF_corr": {"index": 1, "biased_color": biased_colors[1], "unbiased_color": unbiased_colors[1], "lw": 3, "ls": '-'},
    "coh_20_choice_toRF_corr": {"index": 2, "biased_color": biased_colors[2], "unbiased_color": unbiased_colors[2], "lw": 3, "ls": '-'},
    "coh_50_choice_toRF_corr": {"index": 3, "biased_color": biased_colors[3], "unbiased_color": unbiased_colors[3], "lw": 3, "ls": '-'},
    "coh_0_choice_awayRF_corr": {"index": 4, "biased_color": biased_colors[0], "unbiased_color": unbiased_colors[0], "lw": 3, "ls": '--'},
    "coh_6_choice_awayRF_corr": {"index": 5, "biased_color": biased_colors[1], "unbiased_color": unbiased_colors[1], "lw": 3, "ls": '--'},
    "coh_20_choice_awayRF_corr": {"index": 6, "biased_color": biased_colors[2], "unbiased_color": unbiased_colors[2], "lw": 3, "ls": '--'},
    "coh_50_choice_awayRF_corr": {"index": 7, "biased_color": biased_colors[3], "unbiased_color": unbiased_colors[3], "lw": 3, "ls": '--'},
}
for neuron in neuron_metadata.neuron_id:
    session_id = neuron_metadata.session_id[neuron_metadata.neuron_id == neuron].values[0]

    if session_id not in session_metadata["session_id"].values:
        continue
    plot_PSTH(neuron, condition_dict, alignment_dict=ephys_config["alignment_settings_GP"])
